# 03 — Federator aggregation

`Federator` extends `Mapper` with a fixed reduction op. Under the hood it
builds two `Mapper` scarlets from one base name: `{name}_mapper_reducer`
(where each worker's local contribution lands) and `{name}_mapper_global`
(where the aggregated result is stored, under the fixed key `"global"`).

Since `Federator` *is* a `Mapper` bound to the reducer namespace, workers
post their contribution with the ordinary inherited `Map(value, key=...)` -
one call per worker, no coordination between them. Once enough contributions
are in, `Aggregate(seed)` folds everything currently in the reducer with the
fixed op and writes the result to the global Mapper.

In [ ]:
import os
import numpy as np
from scarlets.core.Mapper import Mapper
from scarlets.formulations.Federator import Federator
from scarlets.utils.ScarletUtils import redisConnect

os.environ.setdefault("REDIS_HOST", "localhost")
os.environ.setdefault("REDIS_PORT", "6379")
os.environ.setdefault("REDIS_AUTH_TOKEN", "")
os.environ.setdefault("APP_ID", "notebook_worker")

FED_NAME = "tutorial_federator"

## Cleanup

In [ ]:
def cleanup():
    r = redisConnect()
    for base in (f"{FED_NAME}_mapper_reducer", f"{FED_NAME}_mapper_global"):
        for pattern in (f"{base}_key-value:*", f"{base}_key-list"):
            keys = list(r.scan_iter(match=pattern))
            if keys:
                r.delete(*keys)

cleanup()
print("cleaned up")

## Workers post their contributions

Three simulated workers, each with their own `Federator` handle on the same
`FED_NAME`. `Map` is inherited from `Mapper` - it writes into the reducer
namespace under each worker's own key.

In [ ]:
worker_values = [np.array([1.0, 2.0]), np.array([3.0, 4.0]), np.array([5.0, 6.0])]

for i, value in enumerate(worker_values):
    fed = Federator(FED_NAME, op=Mapper.SUM)
    fed.Map(value, key=f"worker_{i}")
    print(f"worker_{i} posted {value}")

## Aggregating

Any `Federator` handle on `FED_NAME` can now aggregate - `Aggregate` folds
every contribution currently in the reducer with `op`, starting from the
given seed, and writes the result to the global Mapper under `"global"`.

In [ ]:
fed_head = Federator(FED_NAME, op=Mapper.SUM)
global_result, ok, exc = fed_head.Aggregate(np.zeros(2))
global_result

## Reading the global result

`mpr_global` is the aggregated-result Mapper - any node can read it back
without re-running the aggregation.

In [ ]:
fed_head.mpr_global.AllGather()

## Note: rounds aren't automatic

The reducer isn't cleared after `Aggregate` - a second `Aggregate` call
right now would fold the same three contributions again. Starting a new
round means explicitly resetting the reducer first: `fed_head.resetAll(...)`
or `fed_head.clearAll()` (both inherited from `Mapper`, since `Federator`
*is* the reducer Mapper).

## Cleanup (teardown)

In [ ]:
cleanup()
print("cleaned up")